# Genome Search: interactive notebook
High-performance genomic sequence search, dependency-aware workflow scheduling, concurrent execution and relational storage. Run from the `notebooks/` folder of the repository; it imports the engine from `../app/engine`.
Contact: harshinisuresha7@gmail.com

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../app"))   # the engine lives in app/engine

## 0 · Setup
The `engine` package holds the system: sequence tools and k-mer index (`seqtools`), alignment (`align`), the workflow scheduler and execution engine (`workflow`), the relational store (`store`) and benchmarks (`bench`). This notebook walks through each part with your own data.

In [ ]:
import os, sys, time, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Patch
from engine import seqtools as sq, align as al, workflow as wf, bench as bn
from engine.store import Store

BLUE, ORANGE, GREEN, SLATE, PLUM = "#0b63b5", "#e07a1f", "#1c8c6e", "#5b6b7c", "#8a4fb0"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
print("CPUs on this machine:", os.cpu_count())

## 1 · Data
Generate synthetic sequences (some related to sequence 1, with a planted motif) and store them in the relational schema. To use your own data, see section 8.

In [ ]:
seqs = sq.synthetic(n=12, length=2000, gc_target=.5, motif="GATTACAGATTACA", related=.4, seed=1)
store = Store()
store.add_sequences("synthetic", seqs, sq.kind_of, sq.gc)
df = store.df("SELECT name, kind, length, ROUND(gc, 3) AS gc FROM sequences ORDER BY seq_id")
display(df)
fig, ax = plt.subplots(1, 2, figsize=(9, 2.8))
ax[0].hist(df.length, bins=10, color=BLUE); ax[0].set(title="Length distribution", xlabel="Length", ylabel="Sequences")
ax[1].scatter(df.length, df.gc, color=ORANGE); ax[1].set(title="GC against length", xlabel="Length", ylabel="GC")
plt.tight_layout(); plt.show()

## 2 · Search
### 2.1 K-mer index against a full scan
The index maps every k-mer to the places it occurs. A query becomes one lookup plus verification. We check that it returns exactly what a scan returns.

In [ ]:
k = 8
idx, build_s, peak_mb = sq.build_index(seqs, k)
print(idx.stats(), f"| build {build_s*1000:.1f} ms, peak {peak_mb:.1f} MB")

query = "GATTACAGATTACA"
t = time.perf_counter(); hits = idx.exact(query); t_index = (time.perf_counter() - t) * 1000
t = time.perf_counter(); ref = [(n, p) for n, s in seqs.items() for p in sq.naive_find(s, query)]; t_scan = (time.perf_counter() - t) * 1000
assert sorted(hits) == sorted(ref), "index and scan disagree"
print(f"{len(hits)} matches | index {t_index:.4f} ms | scan {t_scan:.4f} ms | scan is {t_scan/max(t_index,1e-9):.0f}x slower")

names = list(seqs); fig, ax = plt.subplots(figsize=(9, 0.35 * len(names) + 1))
for i, n in enumerate(names): ax.hlines(i, 0, len(seqs[n]), color="#c9d8e4", lw=6)
ax.scatter([p for _, p in hits], [names.index(n) for n, _ in hits], marker="|", s=250, color=ORANGE, zorder=3)
ax.set_yticks(range(len(names)), names); ax.invert_yaxis(); ax.set(title="Where the query matches", xlabel="Position"); plt.show()

### 2.2 Search with mismatches (seed and vote)
Every query k-mer votes for a start position; candidates are verified by Hamming distance. A guarantee of finding every match holds when the number of k-mers exceeds `d × k`.

In [ ]:
true_seq, true_pos = "seq_001", 100
q = seqs[true_seq][true_pos:true_pos + 30]
mq = sq.mutate(q, .06, random.Random(1), indel=0); d = sq.hamming(q, mq)
rows = idx.approx(mq, d)
print(f"query has {d} mismatches; guaranteed to find all matches: {idx.guaranteed(len(mq), d)}")
display(pd.DataFrame(rows, columns=["sequence", "position", "mismatches", "seed votes"]).head(10))

### 2.3 Trie prefix search
A trie stores shared prefixes once. The chart shows where it branches fully and where it splits into single chains.

In [ ]:
trie = sq.Trie()
for n, s in seqs.items(): trie.insert(s[:24], n)
prefix = seqs["seq_001"][:8]
print("sequences starting with", prefix, "->", trie.prefix(prefix), "| trie nodes:", trie.nodes)
level, per = [trie.root], []
for depth in range(1, 25):
    level = [v for nd in level for kk, v in nd.items() if kk != "$"]; per.append(len(level))
plt.figure(figsize=(6, 2.6)); plt.plot(range(1, 25), per, marker="o", color=BLUE); plt.title("Trie nodes per depth"); plt.xlabel("Depth"); plt.ylabel("Nodes"); plt.show()

## 3 · Alignment
Smith-Waterman (local) fills a scoring matrix and traces back from the best cell. The orange path is the alignment.

In [ ]:
A, B = seqs["seq_001"][:200], seqs["seq_002"][:200]
r = al.align(A, B, mode="local", match=2, mismatch=-1, gap=-2)
print(f"score {r['score']} | identity {r['identity']*100:.0f}% | gaps {r['gaps']} | length {r['length']}")
print("\n".join(s[:100] for s in (r["a"], r["mid"], r["b"])))
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].imshow(r["H"], cmap="Blues", aspect="auto"); ax[0].plot([p[1] for p in r["path"]], [p[0] for p in r["path"]], color=ORANGE, lw=2)
ax[0].set(title="Scoring matrix with traceback", xlabel="Position in B", ylabel="Position in A"); ax[0].grid(False)
w = 9; tb = {}
for j in range(len(B) - w + 1): tb.setdefault(B[j:j+w], []).append(j)
pts = [(i, j) for i in range(len(A) - w + 1) for j in tb.get(A[i:i+w], ())]
ax[1].scatter([j for _, j in pts], [i for i, _ in pts], s=6, color=BLUE); ax[1].invert_yaxis(); ax[1].set(title=f"Dot plot (word size {w})", xlabel="B", ylabel="A")
plt.tight_layout(); plt.show()

## 4 · Workflow engine
Each sequence becomes a small graph of tasks (load, k-mer spectrum, alignment, motif scan) that feeds one report. Kahn's algorithm orders the graph and detects cycles; a priority queue releases ready tasks; a resource budget caps heavy work in flight; a worker pool executes them for real.

In [ ]:
tasks = wf.build_pipeline(seqs, k=8, motif="GATTACA", reps=100, align_len=300)
order, level = wf.toposort(tasks)
print(f"{len(tasks)} tasks in {max(level.values()) + 1} dependency levels; first five in order: {order[:5]}")

try:   # a cycle is refused, not run
    wf.toposort([wf.Task("a", "gc", deps=["b"]), wf.Task("b", "gc", deps=["a"])])
except wf.CycleError as e:
    print("cycle detected:", e)

In [ ]:
COLORS = dict(gc=BLUE, kmer=GREEN, align=ORANGE, motif=PLUM, report=SLATE)

def gantt(res, title):
    fig, ax = plt.subplots(figsize=(10, 1.3 + 0.5 * res["lanes"]))
    for r in res["runs"]:
        crit = r["id"] in res["critical"]
        ax.barh(r["lane"], r["end"] - r["start"], left=r["start"], color=COLORS[r["kind"]], edgecolor="black" if crit else "white", linewidth=1.8 if crit else .5)
    ax.set_yticks(range(res["lanes"]), [f"worker {i+1}" for i in range(res["lanes"])]); ax.invert_yaxis()
    ax.legend(handles=[Patch(color=c, label=k) for k, c in COLORS.items()], ncol=5, loc="upper center", bbox_to_anchor=(.5, -.25), frameon=False)
    ax.set(title=title, xlabel="Milliseconds since start"); plt.show()

workers, mode = 4, "threads"          # try "processes" too
res = wf.run_workflow(tasks, workers=workers, mode=mode, capacity=workers + 2)
print(f"wall {res['makespan']:.0f} ms | CPU work {res['serial']:.0f} ms | speedup {res['speedup']:.2f}x | workers used {res['lanes']} | peak budget {res['peak_cost']}")
print(f"critical path (black outline): {' -> '.join(res['critical'][:5])} ... ({res['critical_ms']:.0f} ms of CPU time)")
gantt(res, f"Real execution on {res['lanes']} {mode[:-1]} worker(s)")

In [ ]:
rows = []
for w in (1, 2, 4, 8):
    rr = wf.run_workflow(tasks, workers=w, mode="threads", capacity=w + 2); rows.append((w, rr["makespan"]))
sweep = pd.DataFrame(rows, columns=["workers", "wall_ms"])
plt.figure(figsize=(6, 3)); plt.plot(sweep.workers, sweep.wall_ms, marker="o", color=BLUE, label="Measured")
plt.plot(sweep.workers, sweep.wall_ms.iloc[0] / sweep.workers, "--", color=SLATE, label="Perfect scaling")
plt.xlabel("Workers"); plt.ylabel("Wall time (ms)"); plt.legend(); plt.title(f"Scaling on this machine ({os.cpu_count()} CPU)"); plt.show()
print("Pure-Python tasks hold the global interpreter lock, so threads rarely scale. Re-run with mode='processes' on a multi-core machine.")

### Synchronization: why shared state needs a lock
Each thread reads a counter, yields, then writes back read+1. Without mutual exclusion, updates are lost.

In [ ]:
no_lock, expected, t1 = wf.counter_demo(8, 1000, use_lock=False)
locked, _, t2 = wf.counter_demo(8, 1000, use_lock=True)
print(f"expected {expected:,} | without lock {no_lock:,} ({expected-no_lock:,} lost updates) | with lock {locked:,} ({t2/t1:.1f}x the time)")

## 5 · Relational store
Every workflow run is saved: jobs, their executions and their results, in one transaction. The query plan shows what the database does before and after indexing.

In [ ]:
run_id = store.record_run(res["runs"], store.seq_ids())
display(store.df("""
SELECT substr(j.task, 1, instr(j.task, ':') - 1) AS kind, COUNT(*) AS jobs, ROUND(SUM(e.finished_ms - e.started_ms)) AS total_ms
FROM jobs j JOIN executions e USING(job_id) WHERE instr(j.task, ':') > 0 GROUP BY kind ORDER BY total_ms DESC"""))
sql = "SELECT * FROM sequences WHERE sample_id = 1"
print("without index:", store.explain(sql)); store.set_indexes(True); print("with index:   ", store.explain(sql))

In [ ]:
exp = store.index_experiment(100_000)
pv = exp.pivot(index="query", columns="indexed", values="ms")[["No indexes", "With indexes"]]
ax = pv.plot.bar(logy=True, color=[SLATE, BLUE], figsize=(6, 3), rot=0); ax.set(ylabel="ms per query (log)", title="Indexes at 100,000 rows", xlabel=""); plt.show()
print((pv["No indexes"] / pv["With indexes"]).round(0).astype(int).astype(str) + "x faster with indexes")

## 6 · Benchmarks on this machine
Search and alignment cost against size, measured here. Each search run also asserts that the scan and the index return identical matches.

In [ ]:
s = bn.bench_search([10_000, 30_000, 100_000, 300_000], qlen=24, k=11, reps=3)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for m, g in s.groupby("method"): ax[0].plot(g.n, g.ms, marker="o", label=m)
ax[0].set(xscale="log", yscale="log", title="Search cost against reference size", xlabel="Bases", ylabel="ms"); ax[0].legend(fontsize=8)
a = bn.bench_align([50, 100, 200, 400])
for m, g in a.groupby("method"): ax[1].plot(g.length, g.ms, marker="o", label=m)
ax[1].set(title="Alignment time against length", xlabel="Length (both sequences)", ylabel="ms"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
display(s.pivot(index="n", columns="method", values="ms").round(4))

## 7 · What the numbers mean
Scans grow with the reference while a lookup stays flat, so the index repays its build after a handful of queries. Alignment cost is quadratic. Workflow speedup is bounded by the critical path and, in pure Python threads, by the interpreter lock. Absolute times depend on this machine; the shapes are the result.

## 8 · Use your own data
Paste FASTA text below or read a file, then re-run the sections above with `seqs` replaced.

In [ ]:
fasta = """>my_seq_1
GATTACAGATTACAGGCT
>my_seq_2
GATTACAGATAACAGGCT
"""
# fasta = open("my_sequences.fasta").read()
seqs = sq.parse_fasta(fasta); print({n: len(s) for n, s in seqs.items()})

## 9 · Real example data
Bundled public-domain sequences live in `app/data` (catalogued in `datasets.json`). Load one and re-run the search/alignment sections above with `seqs` replaced.


In [ ]:
import json, pathlib
cat = json.loads(pathlib.Path('../app/data/datasets.json').read_text())
print([(d["name"], d["file"], d["length"], d["gc"]) for d in cat["datasets"]])
fasta = pathlib.Path('../app/data') / cat["datasets"][2]["file"]  # 10 kb demo subset: fast to index
seqs = sq.parse_fasta(fasta.read_text()); print({n: len(s) for n, s in seqs.items()})


---
**Links:** [Website](https://harshini-suresha.github.io/genome-search) · [Streamlit workbench](YOUR_STREAMLIT_URL) · [Source on GitHub](https://github.com/Harshini-Suresha/genome-search)
**Contact:** harshinisuresha7@gmail.com